In [1]:
import pandas as pd
import numpy as np

In [3]:
from scrape_script import StriveDF, XrdDF


c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py:7: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 7 of the file c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(r.text)
c:\Users\brock\OneDrive\Desktop\BYU\Semester 7\STAT_386\Data_acquisition_blog\STAT-386-Data-Acquisition-Code\scrape_script.py:18: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parse

In [4]:
def blank_data_dict():
    data_dict = {
        "name": [],
        "5P": [],
        "5K": [],
        "c.S": [],
        "f.S": [],
        "5H": [],
        "5D": [],
        "2P": [],
        "2K": [],
        "2S": [],
        "2H": [],
        "2D": [],
        "6P": []
    }
    return data_dict

In [5]:
def create_property_given_game_df(game, property):
    data_dict = blank_data_dict()
    
    if game == "Strive":
        data = StriveDF
    elif game == "Xrd":
        data = XrdDF

    for item in data_dict:
        if item == "name":
            data_dict[item] = list(data["name"])
        else:
            data_dict[item] = find_given_move_property(data, [item], property)
    
    new_DF = pd.DataFrame(data_dict)
    return new_DF


In [38]:
def find_given_move_property(df, property):
    result_property = []
    for column in df.columns:
        if column != "name":
            given_attack_property = []
            index = property_index_dict[property]
            for attack in df[column]:
                try:
                    given_attack_property.append(attack[index])
                except TypeError:
                    given_attack_property.append(None)
            result_property.append(given_attack_property)
    return result_property

def compile_move_properties(property):
    GGST_property = find_given_move_property(StriveDF, property)
    zip_strive_property = list(zip(
        GGST_property[0], 
        GGST_property[1], 
        GGST_property[2], 
        GGST_property[3], 
        GGST_property[4], 
        GGST_property[5], 
        GGST_property[6], 
        GGST_property[7], 
        GGST_property[8], 
        GGST_property[9], 
        GGST_property[10], 
        GGST_property[11],
        GGST_property[12]
    ))


    Xrd_property = find_given_move_property(XrdDF, property)
    zip_xrd_property = list(zip(
        Xrd_property[0], 
        Xrd_property[1], 
        Xrd_property[2], 
        Xrd_property[3], 
        Xrd_property[4], 
        Xrd_property[5], 
        Xrd_property[6], 
        Xrd_property[7], 
        Xrd_property[8], 
        Xrd_property[9], 
        Xrd_property[10], 
        Xrd_property[11]
    ))

    property = []
    for character_data in zip_strive_property:
        for character_move in character_data:
            property.append(character_move)
    for character_data in zip_xrd_property:
        for character_move in character_data:
            property.append(character_move)
    return property


In [39]:
data_dict = {
    "attack": [],
    "game": [],
    "attack_type": [],
    "guard": [],
    "startup": [],
    "recovery": [],
    "on_block": []
 
}

attacks = []
game = []
attack_type = []
game_indicator = "Strive"
for df in [StriveDF, XrdDF]:
    for character in df["name"]:
        for column in df.columns:
            if column != "name":
                attacks.append(f"{character}{column} ({game_indicator})")
                game.append(game_indicator)
                attack_type.append(column)
    game_indicator = "Xrd"

property_index_dict = {
    "guard": 0,
    "startup": 1,
    "recovery": 3,
    "on_block": 4
}

print(len(attacks))
data_dict["attack"] = attacks
data_dict["game"] = game
data_dict["attack_type"] = attack_type
data_dict["guard"] = compile_move_properties("guard")
data_dict["startup"] = compile_move_properties("startup")
data_dict["recovery"] = compile_move_properties("recovery")
data_dict["on_block"] = compile_move_properties("on_block")

final_df = pd.DataFrame(data_dict)
final_df


742


,attack,game,attack_type,guard,startup,recovery,on_block
0,Robo-Ky5P (Strive),Strive,5P,All,5,9,-2
1,Robo-Ky5K (Strive),Strive,5K,All,7,11,-2
2,Robo-Kyc.S (Strive),Strive,c.S,All,7,10,+1
3,Robo-Kyf.S (Strive),Strive,f.S,All,10,20,-9
4,Robo-Ky5H (Strive),Strive,5H,All,13,24,-8
...,...,...,...,...,...,...,...
737,Zato-12K (Xrd),Xrd,2K,Low,7,9,+3
738,Zato-12S (Xrd),Xrd,2S,Mid,9,16,-6
739,Zato-12H (Xrd),Xrd,2H,Mid,11,19,-6
740,Zato-12D (Xrd),Xrd,2D,Low,9,15,-7


In [47]:
final_df_cleaned = final_df.copy()
final_df_cleaned.dropna()

final_df_cleaned["guard"] = (
    final_df_cleaned["guard"]
    .str.replace(r"Mid", "All", regex = True)
    .str.replace(r"All.*", "All", regex = True)
    .str.replace(r"High.*", "High", regex = True)
    .str.replace(r"Low.*", "Low", regex = True)
)
final_df_cleaned["startup"] = (
    final_df_cleaned["startup"]
    .str.replace(r"\{*\[.*", "", regex = True)
    .str.replace(r"[^\d+\+]", "", regex = True)
    .astype(float)
)

def recovery_clean(lst):
  if type(lst) == list:
    if len(lst) == 1:
        return float(lst[0])
    elif len(lst) > 1:
        return float(float(lst[0]) + float(lst[1]))

final_df_cleaned["recovery"] = (
    final_df_cleaned["recovery"]
    .str.replace(r"\[.*", "", regex = True)
    .str.replace(r"[^\d+\+]", "", regex = True)
    .str.split("+")
    .apply(recovery_clean)
    .astype(float)
)
final_df_cleaned["on_block"] = (
    final_df_cleaned["on_block"]
    .str.replace(r"\[.*", "", regex = True)
    .str.replace(r"~.*", "", regex = True)
    .str.replace(r"[^\d+\+\-]", "", regex = True)
    .astype(float)
)
final_df_cleaned["plus?"] = (
    final_df_cleaned["on_block"] > 0
)

len(final_df_cleaned)
final_df_cleaned

,attack,game,attack_type,guard,startup,recovery,on_block,plus?
0,Robo-Ky5P (Strive),Strive,5P,All,5.0,9.0,-2.0,False
1,Robo-Ky5K (Strive),Strive,5K,All,7.0,11.0,-2.0,False
2,Robo-Kyc.S (Strive),Strive,c.S,All,7.0,10.0,1.0,True
3,Robo-Kyf.S (Strive),Strive,f.S,All,10.0,20.0,-9.0,False
4,Robo-Ky5H (Strive),Strive,5H,All,13.0,24.0,-8.0,False
...,...,...,...,...,...,...,...,...
737,Zato-12K (Xrd),Xrd,2K,Low,7.0,9.0,3.0,True
738,Zato-12S (Xrd),Xrd,2S,All,9.0,16.0,-6.0,False
739,Zato-12H (Xrd),Xrd,2H,All,11.0,19.0,-6.0,False
740,Zato-12D (Xrd),Xrd,2D,Low,9.0,15.0,-7.0,False


In [50]:
final_df_cleaned.groupby("game").agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    startup_variance = ("startup", "var"),
    recovery_variance = ("recovery", "var"),
    on_block_variance = ("on_block", "var"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    plus_count = ("plus?", "sum")
)


,average_startup,average_recovery,average_on_block,startup_variance,recovery_variance,on_block_variance,fastest_move,slowest_move,plus_count
game,,,,,,,,,
Strive,11.561364,17.552273,-7.245455,40.848162,50.575849,32.345082,5.0,28.0,37
Xrd,9.896667,14.213333,-4.693333,34.026076,47.258685,37.116343,3.0,29.0,57


In [55]:
final_df_cleaned.groupby(["attack_type", "game"]).agg(
    average_startup = ("startup", "mean"),
    average_recovery = ("recovery", "mean"),
    average_on_block = ("on_block", "mean"),
    startup_variance = ("startup", "var"),
    recovery_variance = ("recovery", "var"),
    on_block_variance = ("on_block", "var"),
    fastest_move = ("startup", "min"),
    slowest_move = ("startup", "max"),
    plus_count = ("plus?", "sum")
)

average_startup  average_recovery  average_on_block  \
attack_type game                                                          
2D          Strive        11.235294         18.500000         -8.794118   
            Xrd            9.200000         16.920000         -8.720000   
2H          Strive        14.727273         25.575758        -12.242424   
            Xrd           12.680000         22.040000         -9.280000   
2K          Strive         6.029412         11.147059         -2.823529   
            Xrd            6.080000          7.600000         -0.360000   
2P          Strive         5.529412          9.705882         -2.470588   
            Xrd            5.440000          7.000000         -0.040000   
2S          Strive        10.823529         18.558824         -8.470588   
            Xrd            9.440000         15.840000         -5.760000   
5D          Strive        20.000000         25.176471        -15.000000   
            Xrd           26.080000         15.160000         -5.560000   
5H          Strive        13.212121         22.818182         -8.424242   
            Xrd           11.440000         19.000000         -4.920000   
5K          Strive         7.147059         11.058824         -3.441176   
            Xrd            6.760000         10.640000         -3.560000   
5P          Strive         5.470588          8.735294         -1.970588   
            Xrd            5.080000          7.320000         -0.560000   
5[D]        Strive        28.000000         25.176471        -10.000000   
6P          Strive        10.205882         22.235294        -13.441176   
            Xrd           10.840000         18.440000         -8.320000   
c.S         Strive         7.176471         10.558824          1.264706   
            Xrd            6.520000         12.640000         -1.240000   
f.S         Strive        10.882353         19.323529         -8.558824   
            Xrd            9.200000         17.960000         -8.000000   

                    startup_variance  recovery_variance  on_block_variance  fastest_move  \
attack_type game                                                                           
2D          Strive          2.670232           9.772727          10.168449          10.0   
            Xrd             4.500000          53.326667          71.626667           6.0   
2H          Strive         13.142045          62.001894          44.564394          11.0   
            Xrd            13.976667          35.873333          50.293333           7.0   
2K          Strive          0.271836           6.129234           3.907308           5.0   
            Xrd             0.910000           3.916667           4.490000           5.0   
2P          Strive          1.347594           6.153298           5.529412           5.0   
            Xrd             2.590000           7.833333           7.706667           4.0   
2S          Strive          1.907308          11.647950          14.499109           7.0   
            Xrd             7.006667          70.140000          73.606667           6.0   
5D          Strive          0.000000           3.786096           0.000000          20.0   
            Xrd             1.826667          28.556667          23.090000          24.0   
5H          Strive          4.297348          18.840909          14.251894          10.0   
            Xrd             5.006667          27.333333          27.910000           6.0   
5K          Strive          1.159537          10.178253           6.981283           5.0   
            Xrd             1.773333           9.240000           9.756667           4.0   
5P          Strive          0.620321           5.655080           4.574866           5.0   
            Xrd             1.243333           8.726667           8.506667           3.0   
5[D]        Strive          0.000000           3.786096           0.000000          28.0   
6P          Strive          1.501783           6.609626       